# 10_compute_novelty_excluding_family

**Objective:** Compute a same-family-excluded novelty variant. For each focal patent it takes the top-1 (maximum) similarity to prior art in the five-year window after masking out priors from the same DOCDB family, giving the strict nearest-neighbour measure used in the competitive-density validation.

**Inputs:** `universe_emb_norm.f16.bin`; `universe_meta.parquet`; `patent_features_base.parquet`.

**Outputs:** `novelty_features_nofam.parquet` (`lens_id`, `n_priors_nofam`, `novelty_min_nofam`, `user_top1_sim_nofam`).

In [ ]:
# Imports
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

## Paths, configuration and device selection

In [ ]:
# Paths, configuration, and device selection
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
EMB_BIN  = PROC / "universe_emb_norm.f16.bin"
META     = PROC / "universe_meta.parquet"
# Focal patent features are reconstructed from pipeline outputs (validated lossless vs the former
# patent_features_base.parquet: identical lens_id set, docdb_family_id, priority_date, embeddings).
EMB_FOCAL  = PROC / "lens_id_to_embedding.parquet"     # NB 03: consolidated focal embeddings
LENS2FAM   = PROC / "lens_to_familyid.parquet"          # NB 01: lens_id -> docdb_family_id
FAM_DATES  = RAW / "bigquery" / "family_dates.parquet"  # DOCDB-family priority dates
PAT_MASTER = PROC / "patent_master.parquet"             # publication-level dates (RECOVERED fallback)
OUT      = PROC / "novelty_features_nofam.parquet"

DIM             = 1024
WINDOW_YEARS    = 5
CHUNK_FOCAL     = 32
EPOCH = pd.Timestamp("1970-01-01").toordinal()

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")
t0 = time.time()

## Load universe metadata and memmap

In [ ]:
# Load universe metadata (with family ids for masking) and memory-map embeddings
print("\n=== 1) Lade Universe-Metadata ===")
meta = pd.read_parquet(META, columns=["row_idx","docdb_family_id","priority_date_days"])
N_universe = len(meta)
priority_days = meta["priority_date_days"].values
fam_null_mask = meta["docdb_family_id"].isna().values
universe_fam  = meta["docdb_family_id"].fillna(-1).astype("int64").values
NULL_SENTINEL = np.iinfo(np.int32).min
n_dated = (priority_days != NULL_SENTINEL).sum()
print(f"  {N_universe:,} Universe-Patente, mit Datum: {n_dated:,}")
print(f"  universe with family_id: {(~fam_null_mask).sum():,}")

emb_mm = np.memmap(EMB_BIN, dtype=np.float16, mode="r", shape=(N_universe, DIM))

## Load and normalise focal patents

In [ ]:
# Load and L2-normalise focal-patent embeddings (with family ids)
print("\n=== 2) Lade Focal-Patente ===")
# Reconstruct focal features from pipeline outputs: embeddings from NB 03; priority_date =
# DOCDB-family date, with publication-level fallback for in-house RECOVERED patents that
# have no family member (mirrors the universe dating logic in 04_build_prior_art_universe).
_emb = pd.read_parquet(EMB_FOCAL, columns=["lens_id", "embedding"])
_fam = pd.read_parquet(LENS2FAM,  columns=["lens_id", "docdb_family_id"]).drop_duplicates("lens_id")
_dts = pd.read_parquet(FAM_DATES, columns=["docdb_family_id", "family_priority_date"])
_pub = pd.read_parquet(PAT_MASTER, columns=["lens_id", "priority_date"]).rename(columns={"priority_date": "_pub_date"})
focal = (_emb.merge(_fam, on="lens_id", how="left")
             .merge(_dts.rename(columns={"family_priority_date": "priority_date"}), on="docdb_family_id", how="left")
             .merge(_pub, on="lens_id", how="left"))
focal["priority_date"] = pd.to_datetime(focal["priority_date"].fillna(focal["_pub_date"]), errors="coerce")
focal = focal[["lens_id", "docdb_family_id", "embedding", "priority_date"]].drop_duplicates("lens_id").reset_index(drop=True)
n_total = len(focal)
focal_valid = focal[focal["priority_date"].notna()].reset_index(drop=True)
print(f"  total: {n_total:,}, mit priority_date: {len(focal_valid):,}")

focal_valid["priority_year"] = focal_valid["priority_date"].dt.year.astype("int32")
focal_fam = focal_valid["docdb_family_id"].astype("Int64").to_numpy()
focal_fam_np = np.where(pd.isna(focal_fam), -2, focal_fam).astype(np.int64)

focal_emb_f32 = np.stack(focal_valid["embedding"].values).astype(np.float32)
norms = np.linalg.norm(focal_emb_f32, axis=1, keepdims=True)
norms[norms < 1e-8] = 1.0
focal_emb_f32 /= norms
focal_emb_f16 = focal_emb_f32.astype(np.float16)
print(f"  focal embedding shape: {focal_emb_f16.shape}")

focal_t     = torch.from_numpy(focal_emb_f16).to(DEVICE)
focal_fam_t = torch.from_numpy(focal_fam_np).to(DEVICE)

## Compute top-1 similarity per focal year, excluding same-family priors

In [ ]:
# Compute the same-family-excluded top-1 similarity per focal year
print("\n=== 3) Top-1 (exkl. same-family) pro Jahr ===")
years = sorted(focal_valid["priority_year"].unique())
print(f"  focal year range: {years[0]} - {years[-1]} ({len(years)} Jahre)")

results = np.full((len(focal_valid), 2), np.nan, dtype=np.float32)

t_compute = time.time()
for y in tqdm(years, desc="years"):
    win_start = pd.Timestamp(f"{y - WINDOW_YEARS}-01-01").toordinal() - EPOCH
    win_end   = pd.Timestamp(f"{y}-01-01").toordinal() - EPOCH

    mask = (priority_days >= win_start) & (priority_days < win_end) & (priority_days != NULL_SENTINEL)
    W_idx = np.where(mask)[0]
    M = len(W_idx)

    focal_year_mask = focal_valid["priority_year"].values == y
    focal_year_idx = np.where(focal_year_mask)[0]
    K = len(focal_year_idx)

    if M == 0 or K == 0:
        for fi in focal_year_idx:
            results[fi, 0] = 0
        continue

    W_np = np.ascontiguousarray(emb_mm[W_idx])
    W_fam_np = universe_fam[W_idx]
    W_t = torch.from_numpy(W_np).to(DEVICE)
    W_fam_t = torch.from_numpy(W_fam_np).to(DEVICE)

    for c in range(0, K, CHUNK_FOCAL):
        idx_chunk = focal_year_idx[c:c+CHUNK_FOCAL]
        k = len(idx_chunk)
        F_chunk     = focal_t[idx_chunk]
        F_chunk_fam = focal_fam_t[idx_chunk]

        sims = F_chunk @ W_t.T
        same_fam = F_chunk_fam.unsqueeze(1) == W_fam_t.unsqueeze(0)
        sims = sims.masked_fill(same_fam, -1.0)
        n_excluded = same_fam.sum(dim=1)
        n_priors_nofam = (M - n_excluded).to(torch.int64)

        max_sim = sims.max(dim=1).values
        max_np  = max_sim.cpu().numpy()
        np_nofam = n_priors_nofam.cpu().numpy()

        all_masked = np_nofam == 0
        results[idx_chunk, 0] = np_nofam
        results[idx_chunk[~all_masked], 1] = 1.0 - max_np[~all_masked]

        del sims, same_fam, max_sim
    del W_t, W_fam_t
    if DEVICE == "mps":
        torch.mps.empty_cache()

print(f"  compute time: {(time.time()-t_compute)/60:.1f} min")

## Write output

In [ ]:
# Assemble and write the nofam novelty table
print("\n=== 4) Schreibe Output ===")
out_df = pd.DataFrame({
    "lens_id":             focal_valid["lens_id"].values,
    "n_priors_nofam":      pd.Series(results[:, 0]).astype("Int64"),
    "novelty_min_nofam":   results[:, 1],
})
out_df["user_top1_sim_nofam"] = 1.0 - out_df["novelty_min_nofam"]
out_df.to_parquet(OUT, compression="zstd")
print(f"  -> {OUT} ({OUT.stat().st_size/1e6:.1f} MB)")

## Quick statistics

In [ ]:
# Report summary statistics
print("\n=== Quick-Stats ===")
valid = out_df["novelty_min_nofam"].notna()
print(f"  focal with valid nofam top1: {valid.sum():,} / {len(out_df):,}")
print(f"  user_top1_sim_nofam:  mean={out_df.loc[valid,'user_top1_sim_nofam'].mean():.4f}, "
      f"median={out_df.loc[valid,'user_top1_sim_nofam'].median():.4f}, "
      f"std={out_df.loc[valid,'user_top1_sim_nofam'].std():.4f}, "
      f"range=[{out_df.loc[valid,'user_top1_sim_nofam'].min():.4f}, {out_df.loc[valid,'user_top1_sim_nofam'].max():.4f}]")
print(f"\n  total runtime: {(time.time()-t0)/60:.1f} min")